In [ ]:
%pip install --upgrade pydantic
%pip uninstall -y evidently
%pip install evidently
%pip install --upgrade evidently
%pip install shap

In [ ]:
%restart_python

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from mlflow.models.signature import infer_signature
import mlflow
import mlflow.sklearn

# ── Carga del dataset ──────────────────────────────────────────────────────────
# En Databricks subí el CSV a DBFS y usá la ruta: /dbfs/FileStore/student_mental_health_burnout.csv
# Localmente podés usar la ruta relativa: ./student_mental_health_burnout.csv
CSV_PATH = "/dbfs/FileStore/student_mental_health_burnout.csv"

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Preprocesamiento ───────────────────────────────────────────────────────────

# Eliminar columna ID (no aporta como feature)
df = df.drop(columns=["student_id"])

# Columnas categóricas a encodear
cat_cols = ["gender", "course", "year", "stress_level", "sleep_quality", "internet_quality"]

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Target: burnout_level (High=1, Low=0)
df["burnout_level"] = (df["burnout_level"] == "High").astype(int)

X = df.drop(columns=["burnout_level"])
y = df["burnout_level"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Burnout distribution (test):\n{y_test.value_counts(normalize=True).round(3)}")

In [ ]:
# ── Entrenamiento + tracking MLflow ───────────────────────────────────────────

# ⚠️ Cambiá el path por tu usuario de Databricks
EXPERIMENT_PATH = "/Users/tu-email@gmail.com/Itba 2026/RandomForest_StudentBurnout"

param_grid = [
    {"n_estimators": 50,  "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9},
]

mlflow.set_experiment(EXPERIMENT_PATH)

best_auc = 0
best_run_id = None

for params in param_grid:
    with mlflow.start_run() as run:
        clf = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=42,
            n_jobs=-1
        )
        clf.fit(X_train, y_train)

        y_pred  = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]

        acc       = accuracy_score(y_test, y_pred)
        auc       = roc_auc_score(y_test, y_proba)
        cm        = confusion_matrix(y_test, y_pred)
        report    = classification_report(y_test, y_pred, output_dict=True)
        signature = infer_signature(X_test, y_pred)

        mlflow.log_params(params)
        mlflow.log_metric("accuracy",  acc)
        mlflow.log_metric("roc_auc",   auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall",    report["1"]["recall"])
        mlflow.log_metric("f1_score",  report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report,                              "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()},  "confusion_matrix.json")

        print(f"n_estimators={params['n_estimators']} max_depth={params['max_depth']} | acc={acc:.4f} auc={auc:.4f}")

        if auc > best_auc:
            best_auc    = auc
            best_run_id = run.info.run_id

print(f"\nMejor run: {best_run_id} | AUC={best_auc:.4f}")

In [ ]:
# ── Comparación de todos los runs del experimento ─────────────────────────────
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_PATH])
cols = ["params.n_estimators", "params.max_depth",
        "metrics.accuracy", "metrics.roc_auc", "metrics.f1_score"]
print(runs_df[cols].sort_values("metrics.roc_auc", ascending=False).to_string(index=False))

In [ ]:
# ── Data Drift con Evidently ───────────────────────────────────────────────────
import numpy as np
np.float_ = np.float64  # compatibilidad

from evidently import Report
from evidently.presets import DataDriftPreset

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

data_current   = pd.DataFrame(X_test).assign(target=y_test.values, prediction=y_pred, proba=y_proba)

y_train_pred  = clf.predict(X_train)
y_train_proba = clf.predict_proba(X_train)[:, 1]
data_reference = pd.DataFrame(X_train).assign(target=y_train.values, prediction=y_train_pred, proba=y_train_proba)

report = Report(metrics=[DataDriftPreset()])
report.run(current_data=data_current, reference_data=data_reference)
report

In [ ]:
# ── Explainability con SHAP ────────────────────────────────────────────────────
import shap

explainer   = shap.Explainer(clf, X_train)
shap_values = explainer(X_test)

shap.summary_plot(shap_values.values, X_test, feature_names=X_test.columns)